# Week 2 Task 2.4 — Baseline Models

This notebook documents the reproducible baseline experiment implemented by `scripts/week2_baseline_models.py`. It uses only `data/processed/feature_dataset.parquet`, does not sample or remove observations, and does not train boosted models.

In [ ]:
from pathlib import Path
import json, joblib
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
metrics = json.loads((ROOT / 'reports/week2_baseline_metrics.json').read_text())
metrics['status'], metrics['dataset']['rows'], metrics['dataset']['predictor_count']

## Methodology and chronological split

The target is `meter_reading`. Raw `timestamp` is reserved for splitting; `building_id` and `site_id` are excluded because direct identity features encourage memorization and do not transfer to unseen entities. All other engineered predictors are retained (42 total).

The fixed split is training through 2016-08-31, validation from 2016-09-01 through 2016-10-31, and test from 2016-11-01 through 2016-12-31. There is no shuffle. The transformer is fitted only on training rows, so future-period category and scale information cannot leak backward.

In [ ]:
pd.DataFrame(metrics['splits']).T

## Preprocessing pipeline

Numeric predictors use training-only median imputation and standardization. `primary_use` and `meter` use training-only most-frequent imputation plus `OneHotEncoder(handle_unknown='ignore')`. Although scaling is not needed by trees, it preserves split ordering and lets all three models share one controlled feature representation. Each saved sklearn `Pipeline` embeds the fitted transformer.

In [ ]:
pipeline = joblib.load(ROOT / metrics['models']['random_forest']['model_path'])
pipeline

## Model training

The script validates every row and logical key, performs the chronological split, fits Linear Regression, Decision Tree, and Random Forest baselines, measures timings, saves artifacts, and reload-verifies them. Set the flag below only when a deliberate full rerun is wanted; the run processes all 20.2 million observations and is computationally substantial.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    import subprocess, sys
    subprocess.run([sys.executable, str(ROOT / 'scripts/week2_baseline_models.py')], check=True)

## Evaluation

RMSE penalizes extreme misses, MAE represents typical absolute error, and R² measures improvement over predicting the split mean. Validation selects the reference model; the untouched test period supplies the final baseline estimate.

In [ ]:
rows = []
for name, info in metrics['models'].items():
    for split in ('validation', 'test'):
        rows.append({'model': name, 'split': split, **info[split],
                     'training_seconds': info['training_time_seconds'],
                     'inference_seconds': info['inference_time_seconds'][split]})
comparison = pd.DataFrame(rows)
comparison

## Feature importance and coefficient analysis

Tree plots show impurity-based importance. Linear Regression shows absolute standardized coefficient magnitudes. Correlated engineered predictors can destabilize linear coefficients, while impurity importance can favor continuous predictors; neither should be interpreted causally.

In [ ]:
for name, info in metrics['models'].items():
    print(name, pd.DataFrame(info['feature_analysis']).head(10)[['feature', 'value']])
    display(Image(filename=str(ROOT / info['feature_analysis_plot'])))

## Discussion

Random Forest is the selected baseline by validation RMSE. Its small R² indicates that non-identity predictors explain little of the variance created by extreme readings, even though MAE is far smaller than RMSE. Test RMSE rises sharply in November–December, signaling temporal drift and/or rare target extremes. Later work should compare advanced models against this frozen baseline, use rolling-origin evaluation, assess unseen-building performance, add residual and permutation-importance analysis, and introduce lag features only after the prediction horizon and inference-time availability are defined.